In [ ]:
import os
import time
import csv
from datetime import datetime

import telebot
from dotenv import load_dotenv

load_dotenv(override=False)
TOKEN = os.getenv("TELEGRAM_BOT_TOKEN")

if not TOKEN:
    raise RuntimeError(
        "Falta TELEGRAM_BOT_TOKEN. Creá el archivo .env junto al notebook."
    )

bot = telebot.TeleBot(TOKEN, parse_mode=None)

RUTA_CSV = "historial.csv"

def registrar(message):
    existe = os.path.isfile(RUTA_CSV)

    with open(RUTA_CSV, "a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        if not existe:
            writer.writerow(["timestamp", "chat_id", "nombre", "mensaje"])

        writer.writerow([
            datetime.now().isoformat(timespec="seconds"),
            message.chat.id,
            message.from_user.first_name or "sin nombre",
            message.text or f"[{message.content_type}]",
        ])

print("Bot creado en Python. Todavía no está escuchando mensajes.")


Bot creado en Python. Todavía no está escuchando mensajes.


In [ ]:
@bot.message_handler(commands=["start"])
def comando_start(message):
    registrar(message)
    nombre = message.from_user.first_name or "persona"
    texto = (
        f"¡Hola, {nombre}! 👋, soy un bot de ejemplo creado en Python 👾. Me has iniciado 🤖🦾"
    )
    with open("foto_bot_cocinando.jpeg", "rb") as foto:
        bot.send_photo(message.chat.id, photo=foto, caption=texto)
    

@bot.message_handler(commands=["atajos"])
def comando_atajos(message):
    registrar(message)
    texto = (
        "<b>Podes usar estos comandos:</b>\n\n"
        "<em>/start</em>\n\n"
        "<em>/dolar</em>\n\n"
        "<em>/ingredientes</em>\n\n"
        "<em>/historial</em>\n\n"
        "<em>/receta</em>\n\n"

        )
    bot.reply_to(message.chat.id, texto, parse_mode = "HTML")

@bot.message_handler(commands=["ingredientes"])
def comando_ingredientes(message):
    texto = ("Esto todavia no fue implementado")
    bot.reply_to(message, texto)

@bot.message_handler(commands=["dolar"])
def comando_dolar(message):
    registrar(message)
    texto = ("Esto todavia no fue implementado")
    bot.reply_to(message, texto)

@bot.message_handler(commands=["historial"])
def comando_historial(message):
    registrar(message)
    chat_id = message.chat.id

    if not os.path.isfile(RUTA_CSV):
        bot.reply_to(message, "Todavía no hay historial 📭")
        return

    with open(RUTA_CSV, "r", encoding="utf-8") as f:
        filas = list(csv.DictReader(f))

    mias = [f for f in filas if f["chat_id"] == str(chat_id)]
    ultimas = mias[-5:]

    if not ultimas:
        bot.reply_to(message, "No encontré consultas tuyas todavía 📭")
        return

    lineas = [f"• {f['timestamp'][11:16]} — {f['mensaje']}" for f in ultimas]
    texto = "🕐 <b>Tus últimas 5 consultas:</b>\n\n" + "\n".join(lineas)

    bot.send_message(chat_id, texto, parse_mode="HTML")
    
@bot.message_handler(commands=["receta"])
def comando_receta(message):
    registrar(message)
    texto = ("esto todavia no fue implementado")
    bot.reply_to(message, texto)


In [42]:
@bot.message_handler(content_types=["text"])
def mensaje_de_texto(message):
    registrar(message)
    texto = (message.text or "").strip()

    bot.send_chat_action(message.chat.id, "typing")

    if texto.startswith("/"):
        bot.reply_to(message, "No conozco ese comando. Probá con /atajos.")
        return

    bot.reply_to(message, "Hola! podes ver los comandos en el menu o usando /atajos")

In [43]:
from telebot import types

bot.set_my_commands([
    types.BotCommand("start", "Iniciar la conversación"),
    types.BotCommand("atajos", "Ver atajos"),
    types.BotCommand("ingredientes", "Ver ingredientes"),
    types.BotCommand("dolar", "Ver cotización"),
    types.BotCommand("historial", "Ver historial"),
    types.BotCommand("receta", "Pedir recetas")
])

print("Menú de comandos configurado.")


Menú de comandos configurado.


In [ ]:
# Ejecutar esta celda al final. Mantener el notebook abierto mientras se prueba el bot.
bot.remove_webhook()
time.sleep(1)
print("Bot escuchando con polling. Para detenerlo: Interrupt Kernel.")
bot.infinity_polling(
    skip_pending=True,
    timeout=20,
    long_polling_timeout=20,
    allowed_updates=["message"],
)

Bot escuchando con polling. Para detenerlo: Interrupt Kernel.


2026-09-15 12:33:20,829 (__init__.py:1160 MainThread) ERROR - TeleBot: "Infinity polling: polling exited"
2026-09-15 12:33:20,831 (__init__.py:1162 MainThread) ERROR - TeleBot: "Break infinity polling"
